# 01b — Téléchargement des images des splits (à lancer APRÈS 01_make_splits)

```text
Téléchargement COCO.
  python 00_download.py annotations          -> annotations_trainval2017.zip (241 Mo) + extraction des 4 JSON utiles
  python 00_download.py images               -> télécharge UNIQUEMENT les images des 3 splits (après 01_make_splits.py)
Pas besoin des 18 Go de train2017 : on ne prend que ~10 200 images via leur coco_url.
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [6]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [7]:
import argparse, concurrent.futures as cf, time, urllib.request, zipfile
from pathlib import Path
from common import get_logger, load_config, mkdirs, paths, read_json
log = get_logger("download")
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
WANTED = ["captions_train2017.json", "captions_val2017.json", "instances_train2017.json", "instances_val2017.json"]

### `fetch()`

In [8]:
def fetch(url: str, dst: Path, retries: int = 4) -> bool:
    if dst.exists() and dst.stat().st_size > 0:
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    for a in range(retries):
        try:
            tmp = dst.with_suffix(dst.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)
            tmp.rename(dst)
            return True
        except Exception as e:  # noqa
            time.sleep(1.5 * (a + 1))
    log.warning("échec: %s", url)
    return False

### `main()`

In [9]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("what", choices=["annotations", "images"])
    ap.add_argument("--workers", type=int, default=16)
    args = ap.parse_args()
    cfg = load_config(); p = paths(cfg); mkdirs(p)
    ann_dir = p["raw"] / "annotations"

    if args.what == "annotations":
        z = p["raw"] / "annotations_trainval2017.zip"
        log.info("téléchargement des annotations...")
        fetch(ANN_URL, z)
        with zipfile.ZipFile(z) as zf:
            for n in WANTED:
                if not (ann_dir / n).exists():
                    zf.extract(f"annotations/{n}", p["raw"])
        log.info("OK -> %s", ann_dir)
        return

    if not (p["splits"] / "kb.json").exists():
        raise SystemExit("Splits introuvables : exécute d'abord le notebook 01_make_splits (qui a besoin des annotations de 00_download_annotations).")
    todo = []
    for split in ["kb", "val", "test"]:
        rec = read_json(p["splits"] / f"{split}.json")
        for r in rec["images"]:
            todo.append((r["coco_url"], p["raw"] / "images" / r["source"] / r["file_name"]))
    log.info("%d images à récupérer", len(todo))
    ok = 0
    with cf.ThreadPoolExecutor(args.workers) as ex:
        for good in ex.map(lambda t: fetch(*t), todo):
            ok += good
    log.info("téléchargées: %d/%d", ok, len(todo))
    if ok < len(todo):
        raise SystemExit("images manquantes: relancer la commande (reprise automatique)")

## Lancer l'étape

In [10]:
import sys
sys.argv = ["notebook", "images"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

19:08:27 | INFO | 10200 images à récupérer
19:08:28 | INFO | téléchargées: 10200/10200
